<a href="https://colab.research.google.com/github/priyalimbu246/assignments/blob/main/Copy_of_Priya_of_Assignment_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Chemical Applications of Machine Learning (CHEM 4930/5610) - Spring 2026

### Assignment 4 - Deadline 3/6/2026
Points 10

#### General Comments
All figures and graph should have approriate labels on the two axis, and should include a legend with appropriate labels of the different plots.

The notebook should be return in working format. That is, I should be able to reset all the output and re-run all the cells and get the same results as you obtained.

**You should start by saving a copy of the notebook to your Google Drive so you preserve all changes.**

**Please add your name as a suffix to the filname**

**Student Name**: Priya Limbu
**AI usage statement:**
I have used claude and perplexicity for codes to better understand.

### Task 1 - 10 points

In this task, we will consider data from this paper:
- Enhancing Permeability Prediction of Heterobifunctional Degraders Using Machine Learning and Metadynamics-Informed 3D Molecular Descriptors - [DOI:10.1021/acs.jcim.5c01600](https://doi.org/10.1021/acs.jcim.5c01600)

Where the authors consider the Permeability of so-called PROTAC compounds that are large and flexible molecules used in Targeted Protein Degradation.

All the dataset used in the paper, and the code use to obtain the results are given in this following Github repository:
- https://github.com/brykimjh/degrader-permeability-ml3d-metaD  

The specfic dataset that we use 32 PROTAC compounds with measured passive permeability (given in nm/s) and includes 17 2D features calculated by RDKit (see [here](https://github.com/brykimjh/degrader-permeability-ml3d-metaD/blob/main/data/calculate_2d_properties.py) for the script they are calculated), and 3 3D "ensemble" features that are obtained from molecular dynamics simulations as described in the paper.

The target value is the measured passive permeability that is experimentaly measured and given in nm/s.

The dataset can be seen here:
- https://github.com/brykimjh/degrader-permeability-ml3d-metaD/blob/main/outputs/ml_models/model_data.csv

Where the log10 transformed passive permeability is given by `P_appLog`. Note that here the passive permeability has been log transformed in based 10 using `np.log10`.

The 2D features obtained using RDKit are:
```
[
 'Molecular Weight (MW)', 'CharVol (characteristic volume)',
 'Flexibility (number of rotatable bonds / number of bonds)',
 'Number of Heavy Atoms (HA)', 'RingAtoms', 'Halogens', 'HeteroAtoms',
 'RotBonds (NRotB)', 'AllBonds', 'RingCount', 'NumStereo',
 'Fraction of sp3 Carbon Atoms (FSP3)', 'Hydrogen Bond Donors (HBD)',
 'Hydrogen Bond Acceptors (HBA)', 'cLogD^7.4',
 'Topological polar surface area (TPSA)',
 'Total non-polar surface area (TNSA)'
]
```
that includes various standard descriptors/features implemented in RDKit

and the 3D "ensemble" features obtained using MD simulations are
```
[
 'Ensemble_Average_PSA_Chloroform_ANI',
 'Ensemble_Average_Num_IMHB_Chloroform_ANI',
 'Ensemble_Average_RadiusOfGyration_Chloroform_ANI'
]
```
which inludes the average polar surface area (PSA), the average number of intramolecular hydrogen bonds (IMHB), and the average radius of gyration (that measures if the molecule is compact or extended).

#### A)
Perform regression using random forest regression model using the following hyperparameters:
- Number of tree: 100
- Maximum depth of each tree: use the default values
- Maximum number of features for each tree: use the default values

You should use the log10 transformed passive permeability values as target values.

You should perform the classification using 3 different feature set:
- 2D RDKit features
- 3D Ensemble features
- Combined set of 2D and 3D features

For each case, perform cross validation (CV) using a CV strategy of your choice, and obtain the average and standard deviation of metrics that measure the performance. You should select the metrics you wish to use.

Based on your analysis, which feature set gives the best results for the regression? Does these results fit with the results you obtained in Assignment 3?

#### D) - Optional for 2 points
For the best performing feature set from A), repeat the analysis using the measured passive permeability values in nm/s (i.e., the non log10 transformed values). Do you observe any measurable difference in using the direct values or the log10 transformed values?


In [ ]:
# Download dataset

%%bash
dataset_url="https://raw.githubusercontent.com/brykimjh/degrader-permeability-ml3d-metaD/refs/heads/main/data/2d_features.csv"
wget ${dataset_url}
ls

In [ ]:
# Bash script to download all the dataset. Don't worry if you don't understand it
%%bash

url="https://raw.githubusercontent.com/brykimjh/degrader-permeability-ml3d-metaD/refs/heads/main/outputs/ml_models/"
dataset_filename="model_data.csv"

rm -f ${dataset_filename}

wget ${url}/${dataset_filename} &> /dev/null

ls

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.model_selection import cross_validate,ShuffleSplit

In [ ]:
df1 = pd.read_csv("model_data.csv")
df2 = pd.read_csv("2d_features.csv")

In [ ]:
df1

In [ ]:
df2

In [ ]:
features_2d = [
    'Molecular Weight (MW)',
    'CharVol (characteristic volume)',
    'Flexibility (number of rotatable bonds / number of bonds)',
    'Number of Heavy Atoms (HA)',
    'RingAtoms',
    'Halogens',
    'HeteroAtoms',
    'RotBonds (NRotB)',
    'AllBonds',
    'RingCount',
    'NumStereo',
    'Fraction of sp3 Carbon Atoms (FSP3)',
    'Hydrogen Bond Donors (HBD)',
    'Hydrogen Bond Acceptors (HBA)',
    'cLogD^7.4',
    'Topological polar surface area (TPSA)',
    'Total non-polar surface area (TNSA)'
]

features_3d = [
    'Ensemble_Average_PSA_Chloroform_ANI',
    'Ensemble_Average_Num_IMHB_Chloroform_ANI',
    'Ensemble_Average_RadiusOfGyration_Chloroform_ANI'
]

features_combined = features_2d + features_3d

Foe 2D-features

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.model_selection import cross_validate,ShuffleSplit

# Features and target
features = df1[features_2d].values
target_used = df1['P_appLog'].values

model = RandomForestRegressor(n_estimators=100, random_state=42)


scoring =['neg_mean_absolute_error',
          'neg_mean_squared_error',
          'neg_root_mean_squared_error',
          'neg_max_error',
          'r2']

# employ 5-fold CV
scores_fold = cross_validate(
    model,
    features, target_used,
    scoring=scoring,
    cv=5,
    return_train_score=True,
    return_estimator=True,
    return_indices=True
)

NumSplits=100
cv_random = ShuffleSplit(n_splits=NumSplits, test_size=0.20)
scores_random = cross_validate(
    model,
    features, target_used,
    scoring=scoring,
    cv=cv_random,
    return_train_score=True,
    return_estimator=True,
    return_indices=True
)
for s in scoring:
  print("Score: {:s}".format(s))
  print("- Training set: {:s}".format(s))
  print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['train_'+s]),np.nanstd(scores_fold['train_'+s])))
  print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['train_'+s]), np.nanstd(scores_random['train_'+s])))
  print("- Test set: {:s}".format(s))
  print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['test_'+s]),np.nanstd(scores_fold['test_'+s])))
  print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['test_'+s]), np.nanstd(scores_random['test_'+s])))
  print("")


For 3D- features

In [ ]:
# Features and target
features = df1[features_3d].values
target_used = df1['P_appLog'].values

model = RandomForestRegressor(n_estimators=100, random_state=42)


scoring =['neg_mean_absolute_error',
          'neg_mean_squared_error',
          'neg_root_mean_squared_error',
          'neg_max_error',
          'r2']

# employ 5-fold CV
scores_fold = cross_validate(
    model,
    features, target_used,
    scoring=scoring,
    cv=5,
    return_train_score=True,
    return_estimator=True,
    return_indices=True
)

NumSplits=100
cv_random = ShuffleSplit(n_splits=NumSplits, test_size=0.20)
scores_random = cross_validate(
    model,
    features, target_used,
    scoring=scoring,
    cv=cv_random,
    return_train_score=True,
    return_estimator=True,
    return_indices=True
)
for s in scoring:
  print("Score: {:s}".format(s))
  print("- Training set: {:s}".format(s))
  print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['train_'+s]),np.nanstd(scores_fold['train_'+s])))
  print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['train_'+s]), np.nanstd(scores_random['train_'+s])))
  print("- Test set: {:s}".format(s))
  print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['test_'+s]),np.nanstd(scores_fold['test_'+s])))
  print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['test_'+s]), np.nanstd(scores_random['test_'+s])))
  print("")


3) For Combined features

In [ ]:
# Features and target
features = df1[features_combined].values
target_used = df1['P_appLog'].values

model = RandomForestRegressor(n_estimators=100, random_state=42)


scoring =['neg_mean_absolute_error',
          'neg_mean_squared_error',
          'neg_root_mean_squared_error',
          'neg_max_error',
          'r2']

# employ 5-fold CV
scores_fold = cross_validate(
    model,
    features, target_used,
    scoring=scoring,
    cv=5,
    return_train_score=True,
    return_estimator=True,
    return_indices=True
)

NumSplits=100
cv_random = ShuffleSplit(n_splits=NumSplits, test_size=0.20)
scores_random = cross_validate(
    model,
    features, target_used,
    scoring=scoring,
    cv=cv_random,
    return_train_score=True,
    return_estimator=True,
    return_indices=True
)
for s in scoring:
  print("Score: {:s}".format(s))
  print("- Training set: {:s}".format(s))
  print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['train_'+s]),np.nanstd(scores_fold['train_'+s])))
  print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['train_'+s]), np.nanstd(scores_random['train_'+s])))
  print("- Test set: {:s}".format(s))
  print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['test_'+s]),np.nanstd(scores_fold['test_'+s])))
  print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['test_'+s]), np.nanstd(scores_random['test_'+s])))
  print("")


#### Analysis - Best Feature Set

Among the three feature sets, the Combined (2D + 3D) features give the best results for regression:

- Lowest test MAE (Random Splits): -0.353 ± 0.113 (vs -0.434 for 2D, -0.403 for 3D)
- Lowest test RMSE (Random Splits): -0.454 ± 0.152
- Best test R² (Random Splits): 0.055 ± 0.560 (only positive R² among all feature sets)
- Best test MSE (Random Splits): -0.229 ± 0.149

The 3D features alone perform better than 2D features alone on the test set, suggesting that ensemble-based 3D molecular descriptors capture important information about permeability.

2) Optional part

Repeating the analysis of combined features using the measured passive permeability values in nm/s (i.e., the non log10 transformed values).


In [ ]:
df1['P_app'] = 10**df1['P_appLog']

In [ ]:
df1['P_app']

In [ ]:

# Features and target
features = df1[features_combined].values
target_used = df1['P_app'].values

model = RandomForestRegressor(n_estimators=100, random_state=42)


scoring =['neg_mean_absolute_error',
          'neg_mean_squared_error',
          'neg_root_mean_squared_error',
          'neg_max_error',
          'r2']

# employ 5-fold CV
scores_fold = cross_validate(
    model,
    features, target_used,
    scoring=scoring,
    cv=5,
    return_train_score=True,
    return_estimator=True,
    return_indices=True
)

NumSplits=100
cv_random = ShuffleSplit(n_splits=NumSplits, test_size=0.20)
scores_random = cross_validate(
    model,
    features, target_used,
    scoring=scoring,
    cv=cv_random,
    return_train_score=True,
    return_estimator=True,
    return_indices=True
)
for s in scoring:
  print("Score: {:s}".format(s))
  print("- Training set: {:s}".format(s))
  print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['train_'+s]),np.nanstd(scores_fold['train_'+s])))
  print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['train_'+s]), np.nanstd(scores_random['train_'+s])))
  print("- Test set: {:s}".format(s))
  print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['test_'+s]),np.nanstd(scores_fold['test_'+s])))
  print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['test_'+s]), np.nanstd(scores_random['test_'+s])))
  print("")


#### Analysis - Log vs Raw Target

Using raw nm/s values gives significantly worse results than log10-transformed values:

- Test MAE jumps from -0.353(log) to -7.570 nm/s (raw) — nearly 20× larger in absolute terms
- Test R² drops from 0.055 (log) to -0.780(raw) — negative, indicating the model performs worse than simply predicting the mean
- The variance in errors is also much higher with raw values (std of RMSE: 3.171 vs 0.152)

This is a well-known phenomenon in machine learning with skewed distributions: the permeability values span several orders of magnitude (0.1 to 49 nm/s), so the log10 transform compresses this range and makes the learning task more tractable. The log transform also ensures that the model treats relative differences (e.g., 1 vs 10 nm/s) with the same weight as larger absolute differences (e.g., 10 vs 100 nm/s), which is more physically meaningful for permeability.